In [1]:
import numpy as np
import pandas as pd

In [8]:
data_nlp = pd.read_csv("../Output/Pred Input/train_not_grouped_5class.csv")
data_ses = pd.read_csv("../Output/Pred Input/SES_train.csv")

# data_combined = pd.merge(data_nlp, data_ses, on='LSOA21CD', how='left')

# sbert_cols = [str(i) for i in range(0, 384)]
# topic_cols = [col for col in data_nlp.columns if col.startswith("topic_")]
# ses_cols = [col for col in data_ses.columns if col != 'LSOA21CD']

X_ses = data_ses.drop(columns=['LSOA21CD'])
# X_sbert = data_combined.drop(columns=['LSOA21CD', 'year', 'gentrification_class', 'class_int'] + ses_cols + topic_cols)
# X_topic = data_combined.drop(columns=['LSOA21CD', 'year', 'gentrification_class', 'class_int'] + ses_cols + sbert_cols)
# X_nlp = data_combined.drop(columns=['LSOA21CD', 'year', 'gentrification_class', 'class_int'] + ses_cols)
# X_combined = data_combined.drop(columns=['LSOA21CD', 'year', 'gentrification_class', 'class_int'])

In [9]:
import torch

y = torch.load("../Output/Pred Input/y_train_5.pt")
y = y.numpy()

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    X_ses, y, test_size=0.2, random_state=42, stratify=y
)

# from sklearn.utils.class_weight import compute_sample_weight
# sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y)),
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_val)

print("Accuracy:", f"{accuracy_score(y_val, y_pred):.4f}")
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4, zero_division=0))

c:\Users\wbwha\anaconda3\envs\nlp_gen\lib\site-packages\xgboost\training.py:183: UserWarning: [18:22:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.6687
Classification Report:
               precision    recall  f1-score   support

           0     0.8771    0.9428    0.9087       507
           1     0.4602    0.4062    0.4315       128
           2     0.4000    0.3519    0.3744       108
           3     0.4733    0.4306    0.4509       144
           4     0.3304    0.3393    0.3348       112

    accuracy                         0.6687       999
   macro avg     0.5082    0.4941    0.5001       999
weighted avg     0.6526    0.6687    0.6595       999



In [5]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid for GridSearchCV
param_grid = {
    'max_depth': range(3, 15, 2),
    'learning_rate': [0.001, 0.01, 0.1],
    'n_estimators': range(50, 200, 10)
}

# Initialize XGBoost classifier
xgb_clf = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y)),
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

# Run grid search with 3-fold cross-validation
print("Running grid search...")
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=5,
    verbose=1,
    # n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Output the best model and evaluate on validation set
best_model = grid_search.best_estimator_

print("\nBest parameters found:")
print(grid_search.best_params_)

print("\nEvaluating best model on validation set...")
y_pred = best_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))

Running grid search...
Fitting 5 folds for each of 270 candidates, totalling 1350 fits


c:\Users\wbwha\anaconda3\envs\nlp_gen\lib\site-packages\xgboost\training.py:183: UserWarning: [02:58:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\wbwha\anaconda3\envs\nlp_gen\lib\site-packages\xgboost\training.py:183: UserWarning: [02:58:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\wbwha\anaconda3\envs\nlp_gen\lib\site-packages\xgboost\training.py:183: UserWarning: [02:58:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\wbwha\anaconda3\envs\nlp_gen\lib\site-packages\xgboost\training.py:183: UserWarning: [02:58:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "


Best parameters found:
{'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 100}

Evaluating best model on validation set...
Accuracy: 0.9958979846620296
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6211
           1       1.00      0.99      1.00      1558
           2       0.99      0.99      0.99       971
           3       0.99      0.99      0.99      1321
           4       0.99      0.99      0.99      1153

    accuracy                           1.00     11214
   macro avg       0.99      0.99      0.99     11214
weighted avg       1.00      1.00      1.00     11214

